This script further processes the initial 1x1km suitability layers of each hydrogen production technology. The steps are as follows:

1) Add distances to nearest feedstock source, electric substation, and surface water flow source above the threshold flow rate given by GRIDCERF using Postgres. The script used can be found in the postgres folder. This is already done. 

2) Filter out candidates that are colocated with a substation and coal candidates that overlap with their feedstock sources (we ignore any overlap between natural gas plants and their feedstock sources because pipelines are typically located underground). Additionally, filter out candidates that intersect their flowing water source, and standardize the columns for the distance to the feedstock source.

3) Calculate the potential of each technology in each load zone, following the same methodology used to caluclate hydrogen plant potentials. (refer to the postprocessing notebook in the hydrogen siting folder)

Step 2 (filtering):

In [1]:
import geopandas as gpd
from pathlib import Path
import pandas as pd 

electricity_processing_dir = Path.cwd() # pre-processing path

input_candidates_path = electricity_processing_dir / 'candidates_with_dists'

output_candidates_path = electricity_processing_dir.parent / 'inputs' / 'final_candidates'
output_candidates_path.mkdir(exist_ok=True)

# Filter out candidates in each file
for generator_file in input_candidates_path.glob('*gpkg'):
    gen_tech = generator_file.stem

    candidates_gdf = gpd.read_file(generator_file)

    candidates_gdf = candidates_gdf[candidates_gdf['dist_to_substation_meters'] != 0]
    candidates_gdf = candidates_gdf[candidates_gdf['dist_to_surface_flow_meters'] != 0]

    if 'coal' in gen_tech:
        candidates_gdf.rename(columns={'dist_to_coal_meters': 'dist_to_feedstock_meters'}, inplace=True)
        candidates_gdf = candidates_gdf[candidates_gdf['dist_to_feedstock_meters'] != 0]

    elif 'gas' in gen_tech:
        candidates_gdf.rename(columns={'dist_to_pipeline_meters': 'dist_to_feedstock_meters'}, inplace=True)

    candidates_gdf.to_file(output_candidates_path / f'{gen_tech}.gpkg', driver='GPKG')
    print(f'Saved {gen_tech}.gpkg to {output_candidates_path}')

Saved gas_cc.gpkg to /Users/nicholaskong/Desktop/REAM_lab/hydrogen_siting/electricity_siting/inputs/final_candidates
Saved gas_cc_ccs.gpkg to /Users/nicholaskong/Desktop/REAM_lab/hydrogen_siting/electricity_siting/inputs/final_candidates
Saved coal_igcc.gpkg to /Users/nicholaskong/Desktop/REAM_lab/hydrogen_siting/electricity_siting/inputs/final_candidates
Saved coal_igcc_ccs.gpkg to /Users/nicholaskong/Desktop/REAM_lab/hydrogen_siting/electricity_siting/inputs/final_candidates


Step 3 (calculating potentials):

First, group the technology layers by feedstock source and remove overlaps in order of increasing number of candidates (to preserve a higher proportion of candidates from sparser layers)

In [2]:
# Make a helper function to remove overlaps between layers
def remove_overlaps(base_gdfs, top_gdf):
    """
    Removes overlapping features from the top GeoDataFrame 
    where they overlap with the base GeoDataFrame.
    
    Parameters
    - base_gdf : A list of base layers (overlaps from top will be removed here).
    - top_gdf : The top layer (features overlapping base will be removed).
    
    Returns
    A GeoDataFrame consisting of:
    - only the non-overlapping portions of features from top_gdf
    """
    # Start with first base gdf, geometry only
    combined_base = base_gdfs[0][["geometry"]]

    # Overlay the rest, geometry only
    for base_gdf in base_gdfs[1:]:
        combined_base = gpd.overlay(combined_base, base_gdf[["geometry"]], how="union")

    # Spatial join to find overlapping top features
    overlaps = gpd.sjoin(top_gdf, combined_base, how="inner", predicate="intersects")

    # Keep only those NOT in overlaps
    cleaned_top = top_gdf.loc[~top_gdf.index.isin(overlaps.index)]
    
    return cleaned_top

In [3]:
# Read in the gdfs in order of increasing number of candidate sites
coal_ccs_gdf = gpd.read_file(output_candidates_path / 'coal_igcc_ccs.gpkg')
coal_gdf = gpd.read_file(output_candidates_path / 'coal_igcc.gpkg')
gas_ccs_gdf = gpd.read_file(output_candidates_path / 'gas_cc_ccs.gpkg')
gas_gdf = gpd.read_file(output_candidates_path / 'gas_cc.gpkg')

In [5]:
# Create an output path
grouped_output_path =  electricity_processing_dir / 'candidates_by_tech_group'
grouped_output_path.mkdir(exist_ok=True)

# ==================================================================
# Remove overlaps in order of increasing number of candidate sites
# ==================================================================

# The filtered sites for coal and coal ccs will be just that of coal ccs
coal_ccs_gdf.to_file(grouped_output_path / 'coal.gpkg', driver='GPKG')

# Get the combined sites for gas and gas ccs
gas_gdf = remove_overlaps([coal_ccs_gdf], gas_ccs_gdf)
gas_gdf.to_file(grouped_output_path / 'gas.gpkg', driver='GPKG')

Now calculate the potential of each technology per each load zone.

In [6]:
# Define a dictionary mapping each electricity generating technology to its reference nameplate capacity (MW)
ref_capacity = {
    "gas_cc": 1009,
    "gas_cc_ccs": 943.5,
    "coal_igcc": 764.3,
    "coal_igcc_ccs": 707.7,
}

In [15]:
# Create helper function that calculates the potential capacity per load zone for given tech(s) in MW
def calculate_potential(candidates_gdf, tech_names, ref_capacity):
    """
    Inputs: 
    - candidates_gdf: the gdf of candidate sites for the given tech(s)
    - tech_names: the hydrogen production technologies that the layer is for
    - ref_capacity: the reference capacity of the candidate (MW)

    Outputs:
    - df: a df with the potential capacity per tech by load zone, structured with the following columns:
        - LOAD_AREA, gen_tech1, gen_tech2, site_count, potential_MW
    """

    # Count the number of candidate sites in each load zone
    count_by_load_area = candidates_gdf.groupby("LOAD_AREA").size().reset_index(name="site_count")

    for i in range(1, 3):  
        if i <= len(tech_names):
            count_by_load_area[f"gen_tech{i}"] = tech_names[i-1]
        else:
            count_by_load_area[f"gen_tech{i}"] = ""

    # Calculate total potential capacity in each load zone
    count_by_load_area["potential_MW"] = (
        count_by_load_area["site_count"] * ref_capacity // 1
    )

    return count_by_load_area

In [17]:
# Create a running list of potential capacity
output_df = pd.DataFrame()

# Import the .shp file of load zones
load_zones_gdf = gpd.read_file(electricity_processing_dir / 'load_zones' / 'load_zones.shp')

# Load final suitable candidate sites for each technology and calculate potential capacity by load zone
for tech_file in grouped_output_path.glob("*.gpkg"):
    gdf = gpd.read_file(tech_file)

    file_name = tech_file.stem
    
    if file_name == 'coal':
        tech_names = ['coal_igcc', 'coal_igcc_ccs']
        ref_capacity = 707.7
    elif file_name == 'gas':
        tech_names = ['gas_cc', 'gas_cc_ccs']
        ref_capacity = 943.5
    else:
        raise Exception(f'tech name {file_name} not found')

    potential_df = calculate_potential(gdf, tech_names, ref_capacity)
    
    # Append to output DataFrame
    output_df = pd.concat([output_df, potential_df], ignore_index=True)

# Sort 
output_df = output_df.sort_values(by=["LOAD_AREA", "gen_tech1", "gen_tech2"])

# Build the cartesian product of load areas × unique technology sets
load_areas = load_zones_gdf["LOAD_AREA"].unique()

# Get unique tech sets (as tuples) from your output_df
tech_sets = (
    output_df[["gen_tech1", "gen_tech2"]]
    .drop_duplicates()
    .apply(tuple, axis=1)
    .tolist()
)

# Build MultiIndex product of load_areas × tech_sets
all_combinations = pd.MultiIndex.from_product(
    [load_areas, tech_sets],
    names=["LOAD_AREA", "tech_set"]
).to_frame(index=False)

# Expand the tuple back into columns
all_combinations[["gen_tech1", "gen_tech2"]] = pd.DataFrame(
    all_combinations["tech_set"].tolist(), index=all_combinations.index
)
all_combinations = all_combinations.drop(columns="tech_set")

# Merge with output_df on LOAD_AREA + all three prod_tech columns
output_df = all_combinations.merge(
    output_df,
    on=["LOAD_AREA", "gen_tech1", "gen_tech2"],
    how="left"
).fillna(0)

# Save
output_csv_path = electricity_processing_dir.parent / 'inputs' / "gen_tech_potentials.csv"
output_df.to_csv(output_csv_path, index=False)
print(f"Saved technology capacity by load zone to {output_csv_path}")


Saved technology capacity by load zone to /Users/nicholaskong/Desktop/REAM_lab/hydrogen_siting/electricity_siting/inputs/gen_tech_potentials.csv
